
# Phase 5-1: PU-Learning 모델 학습 (동구 데이터 전용)
이 노트북은 1.5%의 극단적인 클래스 불균형(Class Imbalance)을 극복하기 위해 **PU-Bagging Random Forest** 알고리즘을 사용하여 동구(Dong-gu) 지역의 데이터를 기반으로 범용 모델을 훈련합니다.
훈련된 모델은 `models/universal_rf_model.pkl`에 저장되며, 이후 광주시 전체의 핫스팟을 추론(Zero-Shot Inference)하는 데 사용됩니다.


In [1]:

import geopandas as gpd
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

base_dir = Path('../../')
dataset_path = base_dir / 'data/processed/dataset_dong-gu_gwangju_south_korea_10m_buf10m_baseline.gpkg'
model_dir = base_dir / 'data/models'
model_dir.mkdir(parents=True, exist_ok=True)

print("✅ 라이브러리 및 경로 설정 완료")


✅ 라이브러리 및 경로 설정 완료



## 1. 데이터 로드 및 피처(X), 타겟(Y) 분리


In [2]:

print(f"📦 동구 데이터셋 로드 중... ({dataset_path.name})")
gdf = gpd.read_file(dataset_path)

# 결측치 처리 (안전장치)
gdf['is_trash'] = gdf['is_trash'].fillna(0).astype(int)

# 모델에 들어갈 X 피처 컬럼들 자동 추출 (카테고리_count_거리m)
feature_cols = [col for col in gdf.columns if '_count_' in col]
print(f"추출된 피처 목록: {feature_cols}")

X = gdf[feature_cols].values
y = gdf['is_trash'].values

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Positive(쓰레기 O): {sum(y==1):,}개, Unlabeled(모름): {sum(y==0):,}개")


📦 동구 데이터셋 로드 중... (dataset_dong-gu_gwangju_south_korea_10m_buf10m_baseline.gpkg)


추출된 피처 목록: ['nightlife_count_30m', 'nightlife_count_50m', 'nightlife_count_100m', 'cafe_count_30m', 'cafe_count_50m', 'cafe_count_100m', 'food_count_30m', 'food_count_50m', 'food_count_100m', 'retail_count_30m', 'retail_count_50m', 'retail_count_100m', 'service_count_30m', 'service_count_50m', 'service_count_100m']
X shape: (113682, 15), y shape: (113682,)
Positive(쓰레기 O): 1,780개, Unlabeled(모름): 111,902개



## 2. PU-Bagging 앙상블 알고리즘 구현 (Rule 5 준수)


In [3]:

class PUBaggingRandomForest:
    def __init__(self, n_estimators=10, random_state=42):
        self.n_estimators = n_estimators
        self.models = []
        self.random_state = random_state
        
    def fit(self, X, y):
        p_idx = np.where(y == 1)[0]
        u_idx = np.where(y == 0)[0]
        
        np.random.seed(self.random_state)
        
        for i in range(self.n_estimators):
            # Unlabeled 중 Positive 개수만큼만 샘플링 (Under-sampling)
            sampled_u_idx = np.random.choice(u_idx, size=len(p_idx), replace=True)
            
            train_idx = np.concatenate([p_idx, sampled_u_idx])
            X_train = X[train_idx]
            y_train = np.concatenate([np.ones(len(p_idx)), np.zeros(len(sampled_u_idx))])
            
            rf = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=self.random_state+i)
            rf.fit(X_train, y_train)
            self.models.append(rf)
            
    def predict_proba(self, X):
        probas = np.array([model.predict_proba(X)[:, 1] for model in self.models])
        # 여러 트리의 확률값을 산술 평균내어 최종 확률 사출 (Rule 5 정규화 보장)
        return probas.mean(axis=0)

print("✅ PUBaggingRandomForest 클래스 준비 완료")


✅ PUBaggingRandomForest 클래스 준비 완료



## 3. 모델 학습 (Train) 및 직렬화 저장


In [4]:

# 모델 초기화 (10개의 서브 모델 앙상블)
model = PUBaggingRandomForest(n_estimators=10, random_state=42)

print("🏋️‍♂️ 모델 학습(Train)을 시작합니다... (이 작업은 다소 시간이 소요될 수 있습니다)")
model.fit(X, y)
print("✨ 모델 학습 완료!")

# 결과 텐서 검증 (Rule 5)
sample_scores = model.predict_proba(X[:1000])
print(f"샘플 스코어 최소값: {sample_scores.min():.4f}, 최대값: {sample_scores.max():.4f}")
if sample_scores.min() >= 0.0 and sample_scores.max() <= 1.0:
    print("🟩 [PASS] Rule 5를 통과했습니다. 안전한 점수 분포입니다.")

# 모델 직렬화(Serialization) 및 저장 (Rule 4)
model_save_path = model_dir / 'universal_rf_model.pkl'
joblib.dump(model, model_save_path)
print(f"🚀 [완료] 학습된 모델이 성공적으로 저장되었습니다! -> {model_save_path}")


🏋️‍♂️ 모델 학습(Train)을 시작합니다... (이 작업은 다소 시간이 소요될 수 있습니다)


✨ 모델 학습 완료!


샘플 스코어 최소값: 0.2411, 최대값: 0.7688
🟩 [PASS] Rule 5를 통과했습니다. 안전한 점수 분포입니다.


🚀 [완료] 학습된 모델이 성공적으로 저장되었습니다! -> ../../data/models/universal_rf_model.pkl
